In [1]:
#Imports generales
import math
import sys
import numpy as np
import random

#Imports gráficos
import matplotlib.pyplot as plt

#Imports AG
import deap
from deap import base, creator, tools

#Imports AC
import cellpylib as cpl

#Imports paralelización
#import multiprocessing
#from multiprocessing import Pool
from joblib import Parallel, delayed
import time
import os

Parámetros del AC
NUM_STATES -> Número de estados de cada célula
VEC_SIZE -> Vecinos considerados a cada lado de la célula
CA_NUM -> Número de autómatas utilizados en la función de fitness para testear la regla
CA_SIZE -> Tamaño del autómata
CA_TIMESTEPS -> Pasos de evolución de los autómatas

Parámetros del AG
IND_SIZE -> Tamaño de los individuos
POP_SIZE -> Tamaño de la poblacion inicial
CXPB -> Probabilidad de cruce
MUTPB -> Probabilidad de mutacion
NGEN -> Número de generaciones

In [4]:
NUM_STATES = 2 
VEC_SIZE = 3 
CA_NUM = 250
CA_SIZE = 75
CA_TIMESTEPS = 135

IND_SIZE = int(pow(NUM_STATES, 2*VEC_SIZE + 1)) 
POP_SIZE = 200
CXPB, MUTPB, NGEN = 0.9, 0.02, 400
FITNESS_THRESHOLD = 0.75
STOP_CONDITION = 0.97

'''Creador de fitness y de individuo
#Fitness con único objetivo (weights = (1.0,) y máximo'''
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

'''Especificación de genes, individuos y poblacion
Individuos binarios (attr_bool) y en forma de lista
'''
toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)


'''Generación uniforme de autómatas iniciales respecto a la proporcion de unos'''
def uniform_ca(ca_num, ca_size):
    densities = np.linspace(0.0, 1.0, ca_num)
    CAs = []
    for i in range(ca_num):
        numbers = np.random.rand(ca_size)
        ca_binario = (numbers <= densities[i]).astype(int)
        CAs.append(np.expand_dims(ca_binario, axis=0))

    return CAs


'''Creación del diccionario de reglas. Recibe un individuo (array binario), y para cada elemento
calcula su índice en binario (vecindario) y le asigna el nuevo estado de la célula central'''
def gen_rule_dict(individual):
    rule_dict = {}
    for i in range(len(individual)):
        binario = format(i, f'0{2*VEC_SIZE + 1}b')
        rule_dict[binario] = individual[i]

    return rule_dict


'''Creador de regla de transición en el formato requerido por CellPyLib a partir de diccionario de reglas.'''
def create_transition_rule(rule_dict):
    def my_rule(cells, r, t):
        
        key = ''.join(str(int(c)) for c in cells)
        return rule_dict[key]
    return my_rule

'''Evolucion de los autómatas de la lista CAs mediante mi_regla'''
def evolved_CAs(CAs, mi_regla):
    evolved_CAs = []
    for i in range(len(CAs)):
        evolved_CA = cpl.evolve(CAs[i], timesteps=CA_TIMESTEPS, apply_rule=mi_regla, r=VEC_SIZE)
        evolved_CAs.append(evolved_CA)

    return evolved_CAs

'''Funcion que genera uniformemente ca_size autómatas, determina la mayoría inicial para cada autómata, construye la regla asociada 
a individual, evoluciona los autómatas con esa regla y devuelve el valor de fitness asociado (porcentaje de aciertos del estado final)'''
def generate_evolve(ca_num, ca_size, individual):
    #Generación uniforme
    CAs = uniform_ca(ca_num, ca_size)

    majority = [1 if np.sum(ca) >= ca_size/2 else 0 for ca in CAs]

    #Diccionario de reglas
    rule_dict = gen_rule_dict(individual)
    mi_regla = create_transition_rule(rule_dict)

    #Evolucionar autómatas 
    CAs = evolved_CAs(CAs, mi_regla)

    #Estudiamos los resultados
    aciertos = 0
    for i in range(ca_num):
        if np.all(CAs[i][-1] == majority[i]):
            aciertos += 1
    return aciertos / ca_num
    
    

'''Funcion de fitness adaptativa:
Genera 20 autómatas aleatoriamente, determina la regla asociada al individuo y los evoluciona con esa regla. 
Si esa regla acierta, de media, en más del 75% de las células, se repite el proceso con 100 autómátas.
Si no, se devuelve el porcentaje por debajo de 75%. De esta forma, se evita evaluar reglas que no son tan buenas
y se gana eficiencia temporal'''
def evaluate_CA_Majority(individual):
    #Generación uniforme
    res = generate_evolve(CA_NUM//4, CA_SIZE, individual)
    if res < FITNESS_THRESHOLD:
        return (res,)
        
    res = generate_evolve(CA_NUM, CA_SIZE, individual)
    
    return (res,)

''' Configuración de cruce, mutacion, seleccion y función de fitness'''
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb = MUTPB)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate_CA_Majority)

'''Creación de población inicial uniforme. Para cada posible densidad, se generan los unos correspondientes, se rellena el individuo
con ceros consecutivos y finalmente se mezcla aleatoriamente el vector (shuffle)'''
def uniform_pop(pop_size, ind_size):
    pop = []

    densities = np.linspace(0.0, 1.0, pop_size)
    
    for density in densities:
        
        num_ones = int(density * ind_size)
        num_zeros = ind_size - num_ones
        
        rule_bits = [1] * num_ones + [0] * num_zeros
        
        random.shuffle(rule_bits)
        
        ind = creator.Individual(rule_bits)
        pop.append(ind)
        
    return pop

def GA():
    
    pop = uniform_pop(POP_SIZE, IND_SIZE)

    '''Evaluación de toda la población paralelamente mediante la libreria joblib y asignacion de cada fitness a su individuo'''
    print('Iniciando cálculo de fitness', flush = True)
    fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in pop)
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit
    print('Cálculo de fitness terminado', flush = True)

    '''Ejecucion del algoritmo NGEN generaciones'''
    for g in range(NGEN):
        print(f'--- Iniciando Generación {g} ---', flush=True)
        
        '''Selección:
        Dejamos que el 20% de los mejores individuos pasen a la siguiente generación'''
        elites = list(map(toolbox.clone, tools.selBest(pop, k= round(0.2*POP_SIZE))))
        
        '''Hacemos selección por torneo para construir el 80% restante de la próxima generacion'''
        offspring = toolbox.select(pop, len(pop) - round(0.2*POP_SIZE))
        offspring = list(map(toolbox.clone, offspring))

        '''Cruce y mutación'''
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            toolbox.mutate(mutant)
            del mutant.fitness.values

        '''Evaluar paralelamente individuos con fitness inválido, es decir, indiviudos cuyo fitness no se ha calculado todavía.'''
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        '''Nueva población: 
        Construimos la población nueva juntando el 20% de elitismo (elites) con el 80% de selección por torneo (offspring)'''
        pop[:] = elites + offspring

        '''Condición de parada:btenemos las 3 mejores soluciones, y si la mejor de ellas supera un fitness umbral, 
        detenemos el algoritmo y devolvemos las 3 mejores soluciones'''
        top = tools.selBest(pop, 3)    
        print(f"--- Gen {g} Completada: Max Fitness = {top[0].fitness.values[0]:.2f}", flush=True)
        print(top[0])
        if top[0].fitness.values[0] >= STOP_CONDITION: #Estudiar condición de parada
            print(f"Parado en la generación {g} con fitness {top[0].fitness.values[0]}")
            return top

    return tools.selBest(pop, 3)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [ ]:
'''Testeo de las mejores reglas obtenidas por el algoritmo genético sobre 250 autómátas'''
CA_NUM_TEST = 250

'''Funcion que muestra gráficamente la regla'''
def dibujar_regla(individual):
    regla = np.array(individual)
        
    simbolo_0 = "□" 
    simbolo_1 = "■" 
    
    print("--- Catálogo de Reglas del AC (Índice -> Configuración -> Resultado) ---")
    
    for indice in range(IND_SIZE):
        
        # 1. Obtener la configuración
        config_binaria_str = f'{indice:07b}'
        
        # 2. Convertir a símbolos visuales
        config_visual = "".join([simbolo_1 if b == '1' else simbolo_0 for b in config_binaria_str])
        
        # 3. Obtener el resultado
        resultado = regla[indice]
        resultado_visual = simbolo_1 if resultado == 1 else simbolo_0
        
        # 4. Imprimir la línea
        print(f"Índice {indice:03d}:   {config_visual}   ->   {resultado_visual}")
        
if __name__ == "__main__":
    top3 = GA()
    
    for individual in top3:
        dibujar_regla(individual)

        '''Generación de autómatas para el testeo y obtención del estado uniforme final deseado'''
        CAs = uniform_ca(CA_NUM_TEST, CA_SIZE)
        majority = [1 if np.sum(ca) >= CA_SIZE/2 else 0 for ca in CAs]
    
        '''Creación del diccionario de reglas'''
        rule_dict = gen_rule_dict(individual)
        mi_regla = create_transition_rule(rule_dict)
    
        '''Evolución paralela, en este caso sí que se puede hacer, de los autómatas'''
        CAs = Parallel(n_jobs=-1)(delayed(cpl.evolve)(CAs[i], timesteps=CA_TIMESTEPS,apply_rule=mi_regla,r=VEC_SIZE) for i in range(len(CAs)))

        '''Estudio del rendimiento de la regla'''
        aciertos = 0
        for i in range(CA_NUM_TEST):
            if np.all(CAs[i][-1] == majority[i]):
                aciertos = aciertos + 1
                
        porcentaje_exito = (aciertos / CA_NUM_TEST) * 100
        print("\n" + "*"*30)
        print(f"  RENDIMIENTO DE LA REGLA:")
        print(f"  Aciertos: {aciertos} / {CA_NUM_TEST}")
        print(f"  FITNESS:  {porcentaje_exito:.2f}%")
        print("*"*30 + "\n")

        '''Representación gráfica de la evolucion de los autómatas de prueba'''
        if individual == top3[0]:
            for i in range(len(CAs)):
                print("Estado uniforme final deseado: ", majority[i])
                plt.figure(figsize=(8, 4))
                plt.imshow(CAs[i], cmap='binary', interpolation='nearest', aspect='auto')
                plt.xlabel("Celda")
                plt.ylabel("Tiempo")
                plt.title(f"Evolución del autómata CA {i}")
                plt.show()


    PRUEBA 1: prueba a ciegas 
    
    PARÁMETROS:
    Parámetros AG
    IND_SIZE = int(pow(2, 2*VEC_SIZE + 1))
    POP_SIZE = 100
    CXPB, MUTPB, NGEN = 0.84, 0.01, 180

    Parámetros AC
    NUM_STATES = 2
    VEC_SIZE = 3
    CA_NUM = 100
    CA_SIZE = 40
    CA_TIMESTEPS = 80
    
    RESULTADOS:
    Fitness de los mejores cromosomas: 70%, 65%, 63%
    Sobran pasos de evolución, suficiente con 50 o 60
    Cierto patrón a expandir zonas con densidad de cierto color hacia un lado (derecha o izquierda). Negro se ve mejor que blanco?
    Los que no consigue clasificar se queda intercambiando casilla blanca con casilla negra, y hay bastantes falsos positivos negros.
    
    
    PRUEBA 2: autómátas pequeños
    
    PARÁMETROS:
    Parámetros AG
    IND_SIZE = int(pow(NUM_STATES, 2*VEC_SIZE + 1))
    POP_SIZE = 150
    CXPB, MUTPB, NGEN = 0.85, 0.03, 250

    Parámetros AC
    NUM_STATES = 2
    VEC_SIZE = 3
    CA_NUM = 100
    CA_SIZE = 20
    CA_TIMESTEPS = 40

    RESULTADOS:
    Fitness de los mejores cromosomas (simulando la regla con 100 autómátas nuevos): 93%, 90%, 89%
    Siguen sobrando Timesteps, en vez de hacer CA_TIMESTEPS = 2*CA_NUM, quizás mejor CA_TIMESTEPS = 1.5*CA_NUM
    Muy buen rendimiento del algoritmo


    PRUEBA 3: aumentamos tamaño del autómata
    Parámetros AG
    IND_SIZE = int(pow(NUM_STATES, 2*VEC_SIZE + 1))
    POP_SIZE = 180
    CXPB, MUTPB, NGEN = 0.85, 0.03, 320

    Parámetros AC
    NUM_STATES = 2
    VEC_SIZE = 3
    CA_NUM = 125
    CA_SIZE = 50
    CA_TIMESTEPS = 75

    RESULTADOS:
    Fitness de los mejores cromosomas: 97,6%, 97,6%, 96,8%
    En la mayoría siguen sobrando timesteps, pero los hay que requieren unas 50 iteraciones para llegar al estado uniforme, por lo que    
    viene bien dejarles  alguna iteracion más para estudiar si consiguen estabilizarse.
    Gran rendimiento del algoritmo.
    


In [ ]:
'''Regla manual de  Gacs-Kurdymov-Levin (GKL)'''
regla_GKL = [0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,1,
             0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,1,1,1,1,1,
             1,1,1,0,1,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,0,1,1,1,1,1]

'''Mejor regla obtenida en las pruebas con el AG'''
regla_AG = [
    0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0,
    0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1,
    0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1,
    0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1,
    0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1,
    0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1,
    0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1
]

'''Comparamos el rendimiento de ambas reglas con 500 autómátas de distintos tamaños'''
num_automata = 250
CA_SIZE_LIST = [50,75,150]
CA_TIMESTEPS_LIST = [75,110,200]
reglas = [regla_AG, regla_GKL]

'''Evaluamos cada una de las reglas con los autómatas de prueba'''
for individual in reglas:
    dibujar_regla(individual)
    for j in range(len(CA_SIZE_LIST)):
        
        CAs = uniform_ca(num_automata, CA_SIZE_LIST[j])
        majority = [1 if np.sum(ca) >= CA_SIZE_LIST[j]/2 else 0 for ca in CAs]
    
        rule_dict = gen_rule_dict(individual)
        mi_regla = create_transition_rule(rule_dict)

        
        CAs = Parallel(n_jobs=-1)(delayed(cpl.evolve)(CAs[i], timesteps=CA_TIMESTEPS_LIST[j],apply_rule=mi_regla,r=VEC_SIZE) for i in range(len(CAs)))
    
        aciertos = 0
        for i in range(num_automata):
            if np.all(CAs[i][-1] == majority[i]):
                aciertos += 1
        porcentaje_exito = (aciertos / num_automata) * 100

        
        print("\n" + "*"*30)
        print(f"  RENDIMIENTO DE LA REGLA:")
        print(f"  Aciertos: {aciertos} / {num_automata}")
        print(f"  FITNESS:  {porcentaje_exito:.2f}%")
        print("*"*30 + "\n")
    
        for i in range(len(CAs)):
            print("Estado uniforme final: ", majority[i])
            plt.figure(figsize=(8, 4))
            plt.imshow(CAs[i], cmap='binary', interpolation='nearest', aspect='auto')
            plt.xlabel("Celda")
            plt.ylabel("Tiempo")
            plt.title(f"Evolución del autómata CA {i}")
            plt.show()
    

--- Catálogo de Reglas del AC (Índice -> Configuración -> Resultado) ---
Índice 000:   □□□□□□□   ->   □
Índice 001:   □□□□□□■   ->   □
Índice 002:   □□□□□■□   ->   □
Índice 003:   □□□□□■■   ->   □
Índice 004:   □□□□■□□   ->   □
Índice 005:   □□□□■□■   ->   □
Índice 006:   □□□□■■□   ->   ■
Índice 007:   □□□□■■■   ->   □
Índice 008:   □□□■□□□   ->   □
Índice 009:   □□□■□□■   ->   ■
Índice 010:   □□□■□■□   ->   □
Índice 011:   □□□■□■■   ->   □
Índice 012:   □□□■■□□   ->   □
Índice 013:   □□□■■□■   ->   ■
Índice 014:   □□□■■■□   ->   □
Índice 015:   □□□■■■■   ->   □
Índice 016:   □□■□□□□   ->   □
Índice 017:   □□■□□□■   ->   □
Índice 018:   □□■□□■□   ->   ■
Índice 019:   □□■□□■■   ->   ■
Índice 020:   □□■□■□□   ->   □
Índice 021:   □□■□■□■   ->   ■
Índice 022:   □□■□■■□   ->   ■
Índice 023:   □□■□■■■   ->   ■
Índice 024:   □□■■□□□   ->   ■
Índice 025:   □□■■□□■   ->   ■
Índice 026:   □□■■□■□   ->   □
Índice 027:   □□■■□■■   ->   ■
Índice 028:   □□■■■□□   ->   □
Índice 029:   □□■■■□■   ->  

Observamos que, aunque muy ligeramente, la regla GLK supera a la encontrada por el algoritmo genético (prueba 3) en los tres casos.